# 🚀 Урок 3. Аналитическое мышление и методология CRISP-DM в e-commerce

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_00_onboarding/lesson_03_analytical_thinking_crisp_dm.ipynb)

## 📌 Бизнес-контекст и методология
Методология **CRISP-DM** (Cross-Industry Standard Process for Data Mining) — проверенный стандарт ведения аналитических проектов.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(tables_query, conn)
print("📋 Таблицы в нашей базе данных:")
print(tables)

df_customers = pd.read_sql_query("SELECT * FROM customers LIMIT 5;", conn)
df_customers


📋 Таблицы в нашей базе данных:
          name
0     products
1    customers
2       orders
3  order_items


,customer_id,first_name,city,acquisition_channel,registration_date,device
0,1,User_1,Moscow,Organic Search,2025-12-27,Desktop Web
1,2,User_2,Samara,Social Media,2025-04-27,Mobile iOS
2,3,User_3,Saint Petersburg,Organic Search,2026-03-14,Mobile Android
3,4,User_4,Yekaterinburg,Direct,2025-05-23,Mobile Android
4,5,User_5,Novosibirsk,Social Media,2025-03-25,Mobile iOS


### 🎯 Практическое задание 1: Аудит каналов привлечения
Посчитайте распределение клиентов по каналам привлечения (`acquisition_channel`).

In [3]:
### ВАШ КОД ЗДЕСЬ
query = """
SELECT 
    acquisition_channel,
    COUNT(customer_id) AS total_customers
FROM customers
GROUP BY acquisition_channel
ORDER BY total_customers DESC;
"""

df_channels = pd.read_sql_query(query, conn)
print(df_channels)

assert not df_channels.empty, "Датафрейм не должен быть пустым"
assert "acquisition_channel" in df_channels.columns, "Колонка acquisition_channel обязательна"
assert df_channels["total_customers"].sum() > 0, "Должны быть клиенты"
print("✅ Тест успешно пройден: каналы привлечения проанализированы!")


  acquisition_channel  total_customers
0            Paid Ads              211
1    Email Newsletter              203
2        Social Media              200
3              Direct              198
4            Referral              197
5      Organic Search              191
✅ Тест успешно пройден: каналы привлечения проанализированы!


### 💼 Реальный кейс: Перевод бизнес-задачи в аналитическую метрику
Стейкхолдер говорит: *«Наш органический поиск стал хуже работать»*.
* **Верный ответ по CRISP-DM**: фиксируем период (LFL / MoM), декомпозируем на Сессии, CR, AOV, долю новых vs повторных покупателей.